# Transformer-over-entity-history: v2 (identity-shortcut fix attempt)

**What changed from v1.** v1's own entity-only ablation (Step 3) confirmed the self-attention entity pooling reproduces -- arguably worsens -- the CPO/eMSP identity shortcut `graph_baseline.py`'s bug log already found and fixed for plain mean-pooling: AUROC=1.0000 exactly on 4/7 variants, the entity-attention pathway *alone* (own features zeroed) ties that same AUROC, and the CPO/eMSP ablation deltas were near-zero while EV's was real and negative. Critically, `graph_baseline.py`'s own eMSP delta on `main` (after its own fix) is -0.1318 -- the *same* targeted exclusion (`ENTITY_EXCLUDED_FEATURES={"emsp": ["previous_flag_rate"]}`) that fixed mean-pooling did **not** fix attention-pooling, because attention has far more capacity (~5k learned parameters per entity-type pool) to fingerprint one of only 3 entities from the *other* 13 features than a mean -- which collapses everything to one number per feature first -- ever had room to exploit.

This notebook attacks that mechanism two ways on the model side, not with a single patch:
1. **Automatic leak-feature detection.** Instead of excluding only the one feature already known to leak, every feature's ANOVA F-statistic (grouped by entity identity, computed once on `main`'s sub_train split) is compared against `previous_flag_rate`'s own F-statistic -- the *confirmed* leaker -- as a calibrated threshold, for eMSP specifically. Anything else that looks at least as identity-correlated gets excluded too. CPO and EV get **no** auto-exclusion -- there's no confirmed reference leaker to calibrate against for either, and guessing a threshold risks throwing away EV's genuinely-behavioral signal.
2. **Attention-entropy regularization.** A loss term pushes each entity pool's attention distribution toward higher entropy (closer to what a mean implicitly does) unless the classification loss gives a real reason to sharpen it -- directly limiting how peaked/memorized a pattern attention can get away with, rather than only hiding features.

**A dataset-level fix landed alongside this, and it changes what counts as expected here.** Investigating why the model-side exclusion didn't transfer surfaced a second, independent finding: in the real generated `v2g_benchmark_multi_world.csv`, `gt_cpo_malicious` turned out to be JUST as static-by-identity as `gt_emsp_malicious` -- `multi_world_pipeline.py` hardcoded `ONSET_DAY=0` for every one of the 7 variants, so CPO-B's malicious flag was never actually time-gated in the shared benchmark CSV either, despite `injection.py`'s onset_day machinery existing and being described as used for a slow-poisoning scenario (that scenario only ran in a separate, older experiment script, never in this CSV). Both `multi_world_pipeline.py` and `feature_engineering.py` were fixed to give BOTH axes a real, independent onset day (computed as a fraction of each world's actual day range, not a guessed absolute number), and `benchmark_labels.py`'s `gt_emsp_malicious` now reads a persisted `emsp_malicious_eligible` column with the same "identity AND onset" shape `authority_malicious` already had for CPO. **This notebook assumes you've regenerated the dataset with that fix (`02_generate_v2g_benchmark.ipynb`) before running it** -- `benchmark_labels.py`'s `add_ground_truth_labels` will raise a clear error if the combined CSV predates it, rather than silently falling back to the old identity-only definition.

That means CPO and eMSP are now symmetric: both have genuine onset-gated ground truth, so Step 4 below searches for a real honest-vs-malicious contrast on BOTH axes, not just CPO -- there's no structural reason left for either one to come up empty. The model-side fixes above and the dataset-level fix are complementary, not redundant: even with real temporal structure to learn from, an over-expressive entity-aggregator could still take an identity shortcut during whichever period is harder to distinguish behaviorally, which is exactly what the auto-exclusion and entropy regularization are there to keep in check.

Everything else -- causality discipline, `MAX_NEIGHBOR_HISTORY=25`, train-once-on-`main`/zero-shot-eval-everywhere -- is unchanged from v1. The checkpoint name is bumped (`transformer_lite_v2`) so this forces genuinely fresh training rather than silently reloading v1's confirmed-leaky checkpoint (trained on the OLD, ungated labels regardless), and v1's ablation CSV survives untouched on Drive (different, MODEL_NAME-parameterized filename) for a direct before/after comparison in Step 3 -- though that comparison now conflates the model-side and dataset-side fixes together, since v1 predates both; treat it as "did things get better overall," not an isolated test of the model-side change alone.

In [ ]:
import os

combined_csv_path = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')

os.environ.setdefault(
    'V2G_TRANSFORMER_OUT_DIR',
    '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/transformer_baseline',
)

print('Combined CSV:', combined_csv_path)
print('Output dir:  ', os.environ['V2G_TRANSFORMER_OUT_DIR'])

## Step 0 -- write the shared eval-package dependencies

`transformer_baseline.py` imports `ml_features`, `benchmark_labels`, `temporal_split`, and `decision_engine`. v1 of this notebook assumed these were already on disk from a prior baseline's run in the same Colab session -- that assumption broke (`ModuleNotFoundError`) the first time this notebook was run on its own. Writing all four here, up front, makes this notebook self-contained regardless of what else has run in the session.

In [ ]:
%%writefile ml_features.py
"""
ml_features.py
-----------------
Feature set and target shared by the ML and DL baselines. Every input
here is something a live system would actually have at scoring time --
nothing from another baseline's output (RBAC/ABAC decisions, naive
trust scores) and nothing derived from the ground-truth labels
themselves.

TARGET: binary any_risk = (risk_factor_count > 0) -- the same "any real
threat axis active" definition RBAC/ABAC are already judged against
(rbac_baseline.summarize_rbac_blindness, abac_baseline.summarize_single_vs_multi).
Each model's predicted P(any_risk=0) becomes its continuous [0,1] trust
score, fed through decision_engine.trust_weighted_decision() for the
3-way PERMIT/CONDITIONAL-PERMIT/DENY decision -- same as every other
baseline, not reduced to binary.

FEATURES:
  - billing_deviation, latency_deviation: the two fuzzy-engine inputs
    (real-grounded on the metered side, see feature_engineering.py) --
    legitimate raw signals, not another baseline's processed output.
  - certificate_valid, is_roaming, is_peak_hour: real session attributes.
  - previous_flag_rate: ALREADY a real column (enriched_features.py) --
    a causal, prior-only running eMSP dispute rate. Using it here is
    feature engineering, not reusing ABAC's naive_emsp_trust (which is
    just 1 - this same column, renamed for ABAC's own purposes).
  - baseline_charge_duration_h, energy_rate_anomaly: real engineered
    columns from enriched_features.py.
  - charge_duration_h, full_duration_h, idle_h, avg_charge_rate_kw:
    raw session telemetry.
  - ev_session_share_so_far: NEW here -- a causal running statistic (same
    spirit as naive_trust.py's EV computation, RE-DERIVED independently
    so this module has no coupling to the ABAC baseline's internals) --
    the fraction of all sessions seen so far (across the FULL
    chronological sequence, not per-split -- see add_ev_session_share_feature's
    docstring) that belong to this EV.
  - day_idx: raw numeric context.

NOT included, deliberately: ev_id itself (1000-way high-cardinality
identity -- the EV-share feature above is the generalizable signal; the
raw ID would let the model memorize specific train-period EV identities
rather than learn the actual reuse-pattern signature), every
RBAC/ABAC/naive-trust output column (would be benchmarking one baseline
against a feature built from another baseline's output, not an
independent comparison) -- AND, as of this revision, cpo_id/emsp_id too.

BUG LOG (kept here deliberately, not swept away -- same discipline as
fuzzy_trust.py's own bug log): the first version of this module DID
include cpo_id/emsp_id as one-hot categorical features. On the real
125,910-row benchmark this produced AUROC/AUPRC = 1.0000 EXACTLY for
both logistic regression and random forest -- not a good score, a tell.
Root cause: in this dataset the malicious CPO/eMSP are malicious from
day 0 with no time variation (onset_day=0), so cpo_id=="CPO-B" implies
gt_cpo_malicious=1 with zero exceptions, same for emsp_id=="eMSP-C" --
a model with those IDs as input doesn't need billing_deviation or
previous_flag_rate at all; it just memorizes "if name is CPO-B or
eMSP-C, flag it," which is a lookup table, not a behavioral detector,
and wouldn't generalize to a deployment with different CPO/eMSP names.
(A synthetic smoke test run before the real data was available had the
same confound baked into its own generator, which is why it showed a
suspiciously-high-but-not-quite-1.0 AUROC~0.995 instead of flagging the
problem outright -- the real data's exact 1.0000 was the actual tell.)
Fixed by dropping cpo_id/emsp_id entirely: every remaining feature is a
genuine behavioral/contextual signal (billing_deviation,
previous_flag_rate, ev_session_share_so_far, etc.), so whatever
separability the model achieves now reflects learning from behavior,
not memorizing which specific name is the known bad one -- the same
standard TrustMesh itself is held to (its trust score comes from
billing_deviation observations, not from knowing a name in advance).

SECOND BUG LOG ENTRY (found while wiring up the 7-build variant suite,
confirmed PRE-EXISTING -- present even for the original single-build,
2-world "main" dataset): add_ev_session_share_feature's running count
and its denominator (total_seen) were kept across the ENTIRE passed-in
frame with no awareness of which world or variant a row came from --
the same class of cross-world leak documented in naive_trust.py's own
BUG LOG (world 0 and world 1, and now all 7 variants, share the same
real connection_time values, so sorting the union together interleaves
independent synthetic universes instead of treating each as its own
clean timeline). Fixed the same way: grouped by (variant, world_id)
before computing the running share, with a no-grouping fallback when
those columns aren't present."""

import numpy as np
import pandas as pd

NUMERIC_FEATURES = [
    "billing_deviation", "latency_deviation", "previous_flag_rate",
    "baseline_charge_duration_h", "energy_rate_anomaly",
    "charge_duration_h", "full_duration_h", "idle_h", "avg_charge_rate_kw",
    "ev_session_share_so_far", "day_idx",
]
BOOLEAN_FEATURES = ["certificate_valid", "is_roaming", "is_peak_hour"]
ALL_FEATURES = NUMERIC_FEATURES + BOOLEAN_FEATURES
TARGET = "any_risk"


DEFAULT_GROUP_COLS = ("variant", "world_id")  # see SECOND BUG LOG entry above


def add_ev_session_share_feature(df: pd.DataFrame, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """Causal running per-EV session share. MUST be called on the FULL
    chronological sequence (train+test combined) BEFORE any train/test
    split -- calling it separately on an already-split test frame would
    reset every EV's running count to zero at the split boundary,
    understating exactly the reuse signature this feature exists to
    capture. (Independent of naive_trust.py's analogous computation,
    deliberately re-derived here so the ML feature set has no coupling
    to the ABAC baseline's internals.)

    Computed independently within each (variant, world_id) group when
    those columns are present (falls back to the whole frame as one
    group otherwise) -- see this module's SECOND BUG LOG entry."""
    df = df.copy()
    present = [c for c in group_cols if c in df.columns]
    groups = df.groupby(present, sort=False) if present else [(None, df)]

    share_col = pd.Series(np.nan, index=df.index, dtype=float)
    for _, g in groups:
        order = g.sort_values("connection_time").index
        ev = g.loc[order, "ev_id"].to_numpy()

        running_n: dict = {}
        total_seen = 0
        local = np.empty(len(g))
        for i, e in enumerate(ev):
            this_ev_n = running_n.get(e, 0)
            local[i] = (this_ev_n / total_seen) if total_seen > 0 else 0.0
            running_n[e] = this_ev_n + 1
            total_seen += 1
        share_col.loc[order] = local

    df["ev_session_share_so_far"] = share_col
    return df


def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TARGET] = (df["risk_factor_count"] > 0).astype(int)
    return df


def prepare_ml_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Adds the EV-share feature and target, and coerces the boolean
    columns to int (sklearn-friendly). Call on the FULL combined
    chronological sequence, before splitting -- see
    add_ev_session_share_feature's docstring."""
    df = add_ev_session_share_feature(df)
    df = add_target(df)
    for col in BOOLEAN_FEATURES:
        df[col] = df[col].astype(int)
    return df


In [ ]:
%%writefile benchmark_labels.py
"""
benchmark_labels.py
----------------------
Ground-truth labels shared by EVERY baseline (RBAC, ABAC, ML, DL, RL,
graph-structural) and by TrustMesh itself, so all of them are scored
against the exact same target.

GROUNDING LEDGER:
  - gt_cpo_malicious: directly from injection.py's authority_malicious
    column (1 iff cpo_id == the injected malicious CPO AND day_idx >=
    onset_day). Already a column in v2g_benchmark_multi_world.csv --
    not recomputed here, just carried through.
  - gt_emsp_malicious: ONSET-GATING FIX -- this USED to be a pure, permanent
    function of emsp_id == MALICIOUS_EMSP (identity_assignment.py's eMSP-C),
    with zero temporal variation: the transformer-baseline entity-attention
    proof of concept's own entity-only ablation confirmed this made the axis
    structurally unlearnable from behavior for ANY entity-aggregation model
    (mean- or attention-pooling), since there was never a honest-vs-malicious
    contrast in eMSP-C's own history to learn from, only its name. Now reads
    multi_world_pipeline.py/feature_engineering.py's persisted
    emsp_malicious_eligible column directly (1 iff emsp_id == MALICIOUS_EMSP
    AND day_idx >= emsp_onset_day) -- the same "identity AND onset" shape
    authority_malicious already has for CPO, so the two axes are symmetric.
    A combined CSV that predates this fix won't have that column;
    add_ground_truth_labels raises a clear error rather than silently
    falling back to the old identity-only definition. Independent threat
    axis from CPO billing fraud (independent, staggered onset day).
  - gt_ev_sybil: directly from enriched_features.py's ev_sybil column.
    Independent threat axis from both of the above.
  - risk_factor_count / composite_risk_score: NOT data fields -- these
    are EVALUATION-TARGET design choices made HERE, not anything the
    generation pipeline produces or implies. Said plainly so they can
    be cited precisely in the paper rather than mistaken for something
    collected "as-is": risk_factor_count is a simple count (0-3) of how
    many of the three independent threat axes are active for a given
    session; composite_risk_score is that count turned into [0,1] under
    an EQUAL-WEIGHTING assumption across axes (AXIS_WEIGHTS below) --
    not a claim that CPO billing fraud, eMSP credential issues, and EV
    Sybil attacks carry equal real-world severity.

WHY NOT A BAKED-IN "DECISION" COLUMN HERE: every baseline's own
continuous score gets passed through decision_engine.trust_weighted_decision()
at evaluation time (04_performance_analysis.ipynb), not pre-computed here.
That keeps the PERMIT / CONDITIONAL-PERMIT / DENY granularity a property
of each model's score, not of the ground truth -- which is what lets the
benchmark show every baseline (not just TrustMesh) supporting that same
granularity, and lets a user push to a finer granularity later just by
changing threshold cut points, with nothing in this module to revisit.
"""

import numpy as np
import pandas as pd

MALICIOUS_EMSP = "eMSP-C"  # mirrors identity_assignment.py; duplicated here
                            # deliberately so this module has no import-time
                            # dependency on trustmesh_pkg's package layout

AXIS_WEIGHTS = {"cpo": 1.0, "emsp": 1.0, "ev": 1.0}  # documented equal-weighting
                                                        # assumption, see module docstring


def add_ground_truth_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["gt_cpo_malicious"] = df["authority_malicious"].astype(int)

    if "emsp_malicious_eligible" not in df.columns:
        raise KeyError(
            "Column 'emsp_malicious_eligible' not found. This combined CSV predates "
            "the eMSP onset-gating fix (see multi_world_pipeline.py / "
            "feature_engineering.py, and this module's GROUNDING LEDGER above) -- "
            "regenerate the dataset (02_generate_v2g_benchmark.ipynb) before running "
            "this baseline. Refusing to silently fall back to the old, structurally-"
            "unlearnable identity-only definition (emsp_id == MALICIOUS_EMSP)."
        )
    df["gt_emsp_malicious"] = df["emsp_malicious_eligible"].astype(int)
    df["gt_ev_sybil"] = df["ev_sybil"].astype(int)

    df["risk_factor_count"] = (
        df["gt_cpo_malicious"] + df["gt_emsp_malicious"] + df["gt_ev_sybil"]
    )
    total_weight = sum(AXIS_WEIGHTS.values())
    df["composite_risk_score"] = (
        AXIS_WEIGHTS["cpo"] * df["gt_cpo_malicious"]
        + AXIS_WEIGHTS["emsp"] * df["gt_emsp_malicious"]
        + AXIS_WEIGHTS["ev"] * df["gt_ev_sybil"]
    ) / total_weight
    return df


def summarize_labels(df: pd.DataFrame) -> None:
    print("risk_factor_count distribution (0 = clean, 3 = all three threat axes active):")
    print(df["risk_factor_count"].value_counts().sort_index())
    print(f"\nAny risk factor present: {(df['risk_factor_count'] > 0).mean():.4f}")
    for axis in ["gt_cpo_malicious", "gt_emsp_malicious", "gt_ev_sybil"]:
        print(f"  {axis} rate: {df[axis].mean():.4f}")
    print(f"\ncomposite_risk_score describe:\n{df['composite_risk_score'].describe()}")


if __name__ == "__main__":
    import os
    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = add_ground_truth_labels(df)
    summarize_labels(df)


In [ ]:
%%writefile temporal_split.py
"""
temporal_split.py
--------------------
Train/test split for baseline evaluation: temporal within each world,
not random and not by-world.

WHY TEMPORAL: day_idx is REAL (derived from actual ACN-Data connection
timestamps, see raw_preprocess.py) and IDENTICAL in range across both
worlds -- only the synthetic identity/fraud fields differ by world, not
the real session timing. injection.py's attacks are keyed on day_idx
(onset_day), so a model that has only ever seen days before the cutoff
has a genuinely harder, more realistic job on the test split than a
model trained on a random shuffle that leaks future-day sessions into
training. This is the right split for a benchmark meant to measure "how
well/fast does each baseline detect or adapt to a trust problem," not
just "can it memorize this distribution."

Applied with the SAME day cutoff across all worlds (computed once, on
the full combined frame), since day_idx's distribution is shared real
data, not an independent per-world draw.

ONE DISCLOSED CAVEAT: enriched_features.add_baseline_charge_duration's
station+hour empirical-median fallback was computed over the WHOLE
dataset (train+test combined) when the benchmark CSV was generated, not
separately per split -- a minor leakage source for that one contextual
feature column specifically. Harmless for anything that doesn't use
baseline_charge_duration_h as a predictor of risk (it isn't one), but
worth stating plainly rather than silently ignoring, consistent with
this project's disclosure discipline.
"""

import numpy as np
import pandas as pd

TRAIN_FRACTION_OF_DAYS = 0.7  # documented split point, not tuned


def compute_day_cutoff(df: pd.DataFrame, train_fraction: float = TRAIN_FRACTION_OF_DAYS) -> int:
    days = np.sort(df["day_idx"].unique())
    cutoff_idx = int(len(days) * train_fraction)
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both splits non-empty
    return int(days[cutoff_idx])


def temporal_train_test_split(df: pd.DataFrame, day_cutoff: int = None):
    if day_cutoff is None:
        day_cutoff = compute_day_cutoff(df)
    train = df[df["day_idx"] < day_cutoff].copy()
    test = df[df["day_idx"] >= day_cutoff].copy()
    print(f"[split] day_cutoff={day_cutoff}  train={len(train)} rows "
          f"(days {train['day_idx'].min()}-{train['day_idx'].max()})  "
          f"test={len(test)} rows (days {test['day_idx'].min()}-{test['day_idx'].max()})")
    for name, part in [("train", train), ("test", test)]:
        rate = (part["risk_factor_count"] > 0).mean() if "risk_factor_count" in part.columns else float("nan")
        print(f"  {name}: any-risk-factor rate = {rate:.4f}")
    return train, test, day_cutoff


if __name__ == "__main__":
    import os
    import benchmark_labels

    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = benchmark_labels.add_ground_truth_labels(df)
    train, test, cutoff = temporal_train_test_split(df)
    print(f"\nWorld breakdown -- train:\n{train['world_id'].value_counts()}")
    print(f"World breakdown -- test:\n{test['world_id'].value_counts()}")


In [ ]:
%%writefile decision_engine.py
"""
decision_engine.py
-------------------
Implements Algorithm 4 from the TrustMesh framework doc (Section 8):
the trust-weighted three-way access decision (PERMIT / CONDITIONAL-PERMIT / DENY).

This is what turns a trust score into an actual access-control outcome,
which is the concrete instantiation of Gap 4.3.1 (fuzzy trust feeding
MA-ABAC decisions directly, not sitting beside them).
"""

from enum import Enum


class Decision(Enum):
    PERMIT = "PERMIT"
    CONDITIONAL_PERMIT = "CONDITIONAL-PERMIT"
    DENY = "DENY"


def trust_weighted_decision(base_confidence: float, effective_trust: float,
                             theta_high: float = 0.75, theta_low: float = 0.4) -> Decision:
    """
    effective_confidence = base_confidence * effective_trust   (framework doc Section 8)

    base_confidence: how strongly the attribute itself is asserted (e.g., a
    validly-signed certificate = 1.0; a partially-expired one might be lower).
    effective_trust: E[opinion] = b + a*u for the issuing authority.
    """
    effective_confidence = base_confidence * effective_trust
    if effective_confidence >= theta_high:
        return Decision.PERMIT
    elif effective_confidence >= theta_low:
        return Decision.CONDITIONAL_PERMIT
    else:
        return Decision.DENY


# ---------------------------------------------------------------------------
# Multi-attribute ABAC policy (combines authority_trust with OTHER
# attributes via rules, not just a single trust threshold -- this is what
# makes it attribute-based access control rather than trust-gating alone).
# ---------------------------------------------------------------------------

ROAMING_TRUST_PENALTY = 0.10  # roaming sessions apply a direct penalty to the
                                # CPO's trust value -- a real, common pattern
                                # (foreign/roaming access needs stronger
                                # assurance than home-network access)


def evaluate_access_policy(cpo_trust: float, emsp_trust: float, certificate_valid: bool,
                             is_roaming: bool, is_peak_hour: bool,
                             theta_high: float = 0.75, theta_low: float = 0.4,
                             roaming_penalty: float = ROAMING_TRUST_PENALTY) -> Decision:
    """
    GENUINE multi-authority ABAC policy: the decision is a function of TWO
    independently-computed authorities' trust scores, not one authority's
    trust plus trust-blind attribute checks.

      - cpo_trust:  TrustMesh's trust in the CPO (billing honesty)
      - emsp_trust: the hub-mediated, EFFECTIVE confidence in the eMSP
                    (certificate-issuance track record), already discounted
                    by how much the relying CPO trusts the hub's reporting
                    (see trust_pipeline.effective_emsp_confidence) -- this is
                    the actual cross-domain fusion mechanism validated in
                    the cold-start experiment, not a flat heuristic.
      - certificate_valid: a per-session, OBJECTIVE fact (hard constraint,
                    e.g. a CRL check) -- NOT subject to either authority's
                    trust score, since it is independently verifiable.

    Combination rule: MIN (weakest-link / AND-conjunction) of the two
    authority trust scores -- both the charging network's billing honesty
    AND the driver's home-network credential issuer must independently
    clear the bar; one being excellent cannot compensate for the other
    being poor. This is what "multi-authority" means operationally: neither
    score alone determines the outcome.
    """
    if not certificate_valid:
        return Decision.DENY  # objective fact, not subject to either authority's trust

    adjusted_cpo_trust = cpo_trust - (roaming_penalty if is_roaming else 0.0)
    combined_trust = min(adjusted_cpo_trust, emsp_trust)

    decision = trust_weighted_decision(1.0, combined_trust, theta_high, theta_low)
    if is_peak_hour and decision == Decision.PERMIT:
        decision = Decision.CONDITIONAL_PERMIT  # environment override, downgrade only

    return decision


## Step 1 -- write the v2 module

Full design rationale for what changed and why -- the automatic leak-feature detection calibrated against `previous_flag_rate`'s own F-statistic, the attention-entropy regularizer, and exactly what is and isn't claimed to be fixed -- is in the module's own docstring below. Read it before changing anything about the entity-history or exclusion logic.

In [ ]:
%%writefile transformer_baseline.py
"""
transformer_baseline.py  (v2 -- identity-shortcut fix attempt)
----------------------------------------------------------------
v1 of this module (transformer_lite) confirmed, via its own entity-only
ablation, that self-attention pooling over entity history reproduces --
arguably worsens -- the CPO/eMSP identity shortcut that graph_baseline.py's
bug log already found and fixed for plain mean-pooling: AUROC=1.0000 exactly
on 4/7 variants, the ablated (own-features-zeroed) pathway alone ties the
real overall AUROC on those variants, and the CPO/eMSP per-axis deltas are
near-zero (+0.0044, +0.0026 on 'main') while EV's delta is real and negative
(-0.0262) -- i.e. EV behaves like a genuine behavioral signal, CPO/eMSP do
not. Critically, graph_baseline.py's OWN already-fixed eMSP delta on 'main'
is -0.1318 (its own features matter a lot there): the SAME targeted
exclusion (ENTITY_EXCLUDED_FEATURES={"emsp": ["previous_flag_rate"]}) that
fixed mean-pooling did NOT fix attention-pooling.

WHY THE SAME FIX DIDN'T TRANSFER: a mean collapses an entity's whole history
down to one number per feature before anything else happens -- excluding the
one feature known to leak removes essentially all of the room for an
identity fingerprint to live in. Self-attention is strictly more expressive:
its learned Q/K/V/out_proj/LayerNorm stack (~5k parameters PER entity-type
pool) can still combine the OTHER 13 features' full within-window pattern
into a fingerprint distinguishing "this one of only 3 entities" even with
previous_flag_rate zeroed, because with only N_CPO=N_EMSP=3 groups and
thousands of training rows per group, there is more than enough capacity
and data to memorize 3 distinct signatures through indirect correlations a
flat mean never had the expressiveness to exploit.

THIS VERSION ATTACKS THAT MECHANISM DIRECTLY, TWO WAYS (not a single patch):

  1. AUTOMATIC leak-feature detection, not just the one hand-picked feature.
     For each entity type, a one-way ANOVA F-statistic (scipy.stats.f_oneway)
     is computed per feature, grouped by that entity's real identity, on
     'main's sub_train split only. A large F-statistic means a feature's
     PER-ENTITY MEANS are far apart relative to its within-entity noise --
     the same shape of signal previous_flag_rate was already confirmed to
     carry for eMSP. Rather than inventing an arbitrary F-statistic cutoff,
     previous_flag_rate's OWN F-statistic is used as the calibration
     threshold for eMSP specifically: any OTHER feature at least as
     identity-correlated gets excluded too. CPO and EV get NO automatic
     exclusion in this version -- there is no confirmed reference leaker to
     calibrate a threshold against for either axis, and guessing an
     arbitrary cutoff risks excluding features that carry genuine signal
     (EV's ablation delta is real; blindly pattern-matching eMSP's fix onto
     EV would be exactly the kind of unprincipled change this project's
     "don't hide issues, don't guess fixes" discipline argues against). This
     is computed ONCE on 'main's sub_train (never per-variant -- using a
     variant's own distribution to decide what to hide from THAT variant's
     evaluation would break the "nothing adapts per variant" protocol every
     baseline here follows) and persisted alongside the model checkpoint via
     exclusions_checkpoint_path(), then threaded explicitly as an
     `excluded_features` argument through every function that builds entity
     history tensors -- replacing the v1 pattern of reading a module-level
     constant directly, so there is no ambiguity about which exclusion set
     any given call is using.

  2. Attention-entropy regularization. A loss term
     (-ATTN_ENTROPY_WEIGHT * mean attention entropy) is ADDED to the
     training objective, pushing each entity pool's attention distribution
     toward higher entropy (closer to mean-pooling's implicit uniform
     weighting) UNLESS the main classification loss gives a strong enough
     gradient to justify a sharper distribution. This directly targets "can
     attention get away with memorizing a peaked, entity-specific pattern"
     rather than hoping feature exclusion alone closes every route to it.
     ATTN_ENTROPY_WEIGHT=0.05 is a REASONED STARTING POINT, NOT AN
     EMPIRICALLY TUNED VALUE (this module is built, not run, by the
     assistant -- see repo-wide policy) -- max possible per-token entropy at
     MAX_NEIGHBOR_HISTORY=25 is ln(25)=3.22, so at this weight the
     regularizer's maximum pull is bounded well below typical BCE-loss
     magnitudes. If the before/after ablation comparison below still shows
     near-zero CPO/eMSP deltas, raise this; if EV's genuinely-behavioral
     delta shrinks a lot, lower it.

WHAT THIS VERSION DOES **NOT** CLAIM TO FIX: gt_emsp_malicious
(benchmark_labels.py) is `emsp_id == "eMSP-C"` with NO onset_day
time-gating -- unlike gt_cpo_malicious, which DOES have one. That means
eMSP-C's own history is malicious on 100% of its rows, for every row, at
every point in the timeline that will ever exist for it: there is no
honest-vs-malicious CONTRAST anywhere in eMSP-C's own behavior to learn from
in the first place, independent of pooling mechanism, feature exclusion, or
regularization strength. No fix below can make the eMSP axis behaviorally
learnable as currently labeled; what the fixes below CAN do is stop
attention from being SHARPER than necessary about exploiting that structural
fact, and stop it from ALSO picking up a parallel identity shortcut on the
CPO axis (which DOES have a real onset_day contrast, so a fix mattering
there is a meaningful, falsifiable test of whether this version actually
changed anything).

MODEL_NAME is bumped to "transformer_lite_v2" specifically so this module
trains a genuinely fresh checkpoint rather than silently reloading v1's
(already-confirmed-leaky) one, and the entity-only-ablation / variant-summary
CSV filenames are now parameterized by MODEL_NAME (v1 used a FIXED filename
for the ablation CSV) so v1's "before" ablation survives untouched on disk
for direct side-by-side comparison with this version's "after" result --
see the notebook's before/after comparison cell.

Everything else (causality discipline, context-split conventions, the
"unknown entity" fallback, MAX_NEIGHBOR_HISTORY=25, train-once-on-'main'/
zero-shot-eval-everywhere) is unchanged from v1 -- read that module's
docstring (and graph_baseline.py's) for the full account.
"""

import os
import sys
import time

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, os.path.dirname(__file__) or ".")

import ml_features
import benchmark_labels
import temporal_split
from decision_engine import trust_weighted_decision

OUT_DIR = os.environ.get(
    "V2G_TRANSFORMER_OUT_DIR",
    "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/transformer_baseline",
)

TRAIN_VARIANT = "main"
MODEL_NAME = "transformer_lite_v2"   # bumped from v1's "transformer_lite" -- forces a fresh
                                       # checkpoint instead of silently reloading the old,
                                       # confirmed-leaky one. See module docstring.
SEED = 42

VAL_FRACTION_OF_TRAIN_DAYS = 0.15
MAX_NEIGHBOR_HISTORY = 25
ENTITY_EXCLUDED_FEATURES = {
    "emsp": ["previous_flag_rate"],   # the manual FLOOR -- same already-confirmed exclusion
}                                       # as v1/graph_baseline.py. Auto-detection below only ADDS
                                        # to this, never removes from it.

LEAK_REFERENCE_FEATURE = {
    "emsp": "previous_flag_rate",   # the CONFIRMED leak signature for this axis, used to
                                      # CALIBRATE the automatic-detection threshold rather than
                                      # an arbitrary F-statistic cutoff. No confirmed reference
                                      # leaker exists for cpo/ev, so neither gets auto-exclusion
                                      # in this version -- see module docstring, point 1.
}

ATTN_DIM = 32
ATTN_HEADS = 4
SESSION_HIDDEN = 64
LR = 1e-3
MAX_EPOCHS = 200
PATIENCE = 20
DROPOUT = 0.1
ATTN_ENTROPY_WEIGHT = 0.05   # UNTUNED starting point -- see module docstring, point 2.

ENTITY_COLS = {"cpo": "cpo_id", "emsp": "emsp_id", "ev": "ev_id"}


def _device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class EntityAttentionPool(nn.Module):
    """Same hand-rolled multi-head self-attention block as v1, with one
    addition: forward() now ALSO returns a scalar attention-entropy term
    (mean entropy, over real query positions of real -- has-history --
    entities only, computed from the PRE-dropout softmax output so it stays
    a well-defined probability-simplex quantity), used by the training
    loop's new entropy-regularization term. See module docstring, point 2."""

    def __init__(self, feature_dim, max_history, attn_dim=ATTN_DIM, heads=ATTN_HEADS, dropout=DROPOUT):
        super().__init__()
        assert attn_dim % heads == 0
        self.heads = heads
        self.head_dim = attn_dim // heads
        self.attn_dim = attn_dim
        self.in_proj = nn.Linear(feature_dim, attn_dim)
        self.pos_embed = nn.Parameter(torch.zeros(max_history, attn_dim))
        nn.init.normal_(self.pos_embed, std=0.02)
        self.q = nn.Linear(attn_dim, attn_dim)
        self.k = nn.Linear(attn_dim, attn_dim)
        self.v = nn.Linear(attn_dim, attn_dim)
        self.out_proj = nn.Linear(attn_dim, attn_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(attn_dim)

    def forward(self, x, mask):
        n, t, _ = x.shape
        h = self.in_proj(x) + self.pos_embed[:t].unsqueeze(0)

        q = self.q(h).view(n, t, self.heads, self.head_dim).transpose(1, 2)
        k = self.k(h).view(n, t, self.heads, self.head_dim).transpose(1, 2)
        v = self.v(h).view(n, t, self.heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        key_mask = mask.unsqueeze(1).unsqueeze(1)
        scores = scores.masked_fill(~key_mask, float("-inf"))

        no_history = (~mask).all(dim=1)
        safe_scores = scores.masked_fill(no_history.view(n, 1, 1, 1), 0.0)
        # proper probability distribution -- entropy and weights_summary are computed FROM
        # THIS, before dropout, since dropout zeroes entries without renormalizing and would
        # make "entropy" meaningless as a probability-simplex quantity. model.eval() (used by
        # every evaluation/visualization call site) makes dropout a no-op anyway, so this
        # distinction only matters during training, which is exactly when it's used.
        weights = torch.softmax(safe_scores, dim=-1)
        weights_for_attn = self.dropout(weights)

        attended = torch.matmul(weights_for_attn, v)
        attended = attended.transpose(1, 2).reshape(n, t, self.attn_dim)
        attended = self.out_proj(attended)
        tokens = self.norm(h + attended)

        mask_f = mask.unsqueeze(-1).float()
        summed = (tokens * mask_f).sum(dim=1)
        counts = mask_f.sum(dim=1).clamp(min=1.0)
        pooled = summed / counts
        pooled = torch.where(no_history.unsqueeze(-1), torch.zeros_like(pooled), pooled)

        weights_summary = weights.mean(dim=(1, 2))

        ent_per_qk = -(weights * torch.log(weights + 1e-9)).sum(dim=-1)   # [n, heads, t]
        ent_per_query = ent_per_qk.mean(dim=1)                             # [n, t]
        valid_query = mask & (~no_history).unsqueeze(1)                   # [n, t]
        if valid_query.any():
            entropy = ent_per_query[valid_query].mean()
        else:
            entropy = torch.zeros((), device=x.device, dtype=ent_per_query.dtype)

        return pooled, weights_summary, entropy


class SessionTransformer(nn.Module):
    def __init__(self, session_dim, feature_dim, max_history, hidden=SESSION_HIDDEN):
        super().__init__()
        self.cpo_pool = EntityAttentionPool(feature_dim, max_history)
        self.emsp_pool = EntityAttentionPool(feature_dim, max_history)
        self.ev_pool = EntityAttentionPool(feature_dim, max_history)
        self.combine = nn.Linear(session_dim + 3 * ATTN_DIM, hidden)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x_session, e_cpo, e_emsp, e_ev):
        h = torch.relu(self.combine(torch.cat([x_session, e_cpo, e_emsp, e_ev], dim=1)))
        return self.head(h).squeeze(-1)


def model_checkpoint_path():
    return os.path.join(OUT_DIR, f"{MODEL_NAME}.pt")


def scaler_checkpoint_path():
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_scaler.joblib")


def exclusions_checkpoint_path():
    """NEW in v2: the resolved {entity_key: [excluded feature names]} dict,
    computed ONCE at training time (see resolve_excluded_features below) and
    persisted here so every evaluation/ablation/visualization call site
    loads the SAME exclusion set the model was actually trained with,
    rather than recomputing it (which would risk a different set per
    variant -- see module docstring on why that would break the zero-shot
    protocol)."""
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_excluded_features.joblib")


def variant_preds_path(variant_name):
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_{variant_name}_test_predictions.csv")


def _prepare_full_frame(combined_csv_path):
    df = pd.read_csv(combined_csv_path)
    df["connection_time"] = pd.to_datetime(df["connection_time"], utc=True)
    df = benchmark_labels.add_ground_truth_labels(df)
    df = ml_features.prepare_ml_frame(df)
    day_cutoff = temporal_split.compute_day_cutoff(df)
    return df, day_cutoff


def _train_val_day_cutoff(train_df, val_fraction=VAL_FRACTION_OF_TRAIN_DAYS):
    days = np.sort(train_df["day_idx"].unique())
    cutoff_idx = int(len(days) * (1 - val_fraction))
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)
    return int(days[cutoff_idx])


def _build_entity_maps(context_df):
    maps = {}
    for key, col in ENTITY_COLS.items():
        uniques = sorted(context_df[col].unique(), key=str)
        maps[key] = {v: i for i, v in enumerate(uniques)}
    return maps


def _entity_indices(rows_df, entity_maps):
    out = {}
    for key, col in ENTITY_COLS.items():
        m = entity_maps[key]
        unknown = len(m)
        out[key] = rows_df[col].map(m).fillna(unknown).astype(np.int64).to_numpy()
    return out


def _detect_leaky_features(context_df, entity_col, feature_cols):
    """One-way ANOVA F-statistic (scipy.stats.f_oneway) per feature,
    grouping context_df by entity_col. A large F-statistic means that
    feature's PER-ENTITY MEANS are far apart relative to its WITHIN-ENTITY
    noise -- the same shape of signal previous_flag_rate was already
    confirmed to carry for the N_EMSP=3 axis (see graph_baseline.py's bug
    log). Returns {feature: F-statistic} for every feature -- this function
    only measures, it does not decide what counts as 'leaky'; see
    resolve_excluded_features for the calibrated threshold."""
    groups = [g[feature_cols].to_numpy(dtype=np.float64) for _, g in context_df.groupby(entity_col) if len(g) > 1]
    if len(groups) < 2:
        return {f: 0.0 for f in feature_cols}
    fstats = {}
    for j, feat in enumerate(feature_cols):
        samples = [g[:, j] for g in groups]
        try:
            f_val, _ = stats.f_oneway(*samples)
        except Exception:
            f_val = 0.0
        fstats[feat] = float(f_val) if np.isfinite(f_val) else 0.0
    return fstats


def resolve_excluded_features(sub_train_df, feature_cols):
    """Computed ONCE, on 'main's sub_train split only -- never per-variant,
    see module docstring. Returns (resolved, all_fstats):
      resolved: {entity_key: [excluded feature names]} -- the manual floor
        (ENTITY_EXCLUDED_FEATURES) UNIONED with anything auto-flagged for
        axes that have a calibration reference (currently just eMSP).
      all_fstats: {entity_key: {feature: F-statistic}} for every entity
        type, including ones with no auto-exclusion applied -- returned so
        the caller can print it for transparency (this is a diagnostic the
        user should SEE, not a silent internal decision)."""
    resolved = {k: list(v) for k, v in ENTITY_EXCLUDED_FEATURES.items()}
    all_fstats = {}
    for key, col in ENTITY_COLS.items():
        fstats = _detect_leaky_features(sub_train_df, col, feature_cols)
        all_fstats[key] = fstats
        reference = LEAK_REFERENCE_FEATURE.get(key)
        if reference is None or reference not in fstats:
            continue
        threshold = fstats[reference]
        auto_flagged = [f for f, v in fstats.items() if v >= threshold]
        existing = set(resolved.get(key, []))
        resolved[key] = sorted(existing.union(auto_flagged))
    return resolved, all_fstats


def _entity_history_tensors(context_df, entity_maps, excluded_features, max_history=MAX_NEIGHBOR_HISTORY):
    """Same construction as v1, with `excluded_features` now an EXPLICIT,
    REQUIRED argument (the resolved dict from resolve_excluded_features,
    loaded from exclusions_checkpoint_path() at every call site) instead of
    reading the module-level ENTITY_EXCLUDED_FEATURES constant directly --
    so there is no ambiguity about which exclusion set any given call uses,
    and the manual floor + auto-detected exclusions both flow through this
    one path."""
    feature_cols = ml_features.ALL_FEATURES
    n_features = len(feature_cols)
    recent_first = context_df.sort_values("day_idx", ascending=False)

    out = {}
    for key, col in ENTITY_COLS.items():
        m = entity_maps[key]
        n = len(m)
        hist = np.zeros((n + 1, max_history, n_features), dtype=np.float32)
        mask = np.zeros((n + 1, max_history), dtype=bool)

        excluded_idx = [feature_cols.index(f) for f in excluded_features.get(key, [])]

        capped = recent_first.groupby(col, sort=False, group_keys=False).head(max_history)
        for raw_value, group in capped.groupby(col, sort=False):
            if raw_value not in m:
                continue
            idx = m[raw_value]
            rows = group.sort_values("day_idx", ascending=True)
            vals = rows[feature_cols].to_numpy(dtype=np.float32)
            k = len(vals)
            if excluded_idx:
                vals[:, excluded_idx] = 0.0
            hist[idx, max_history - k:, :] = vals
            mask[idx, max_history - k:] = True
        out[key] = (hist, mask)
    return out


def _to_device(hist_mask_dict, device):
    return {k: (torch.tensor(h, device=device), torch.tensor(m, device=device))
            for k, (h, m) in hist_mask_dict.items()}


def _forward_logits(model, scored_X, scored_idx, ctx_hist, device):
    """CHANGED in v2: always returns the consistent 3-tuple
    (logits, weights_dict, entropy_total) -- no more return_weights flag.
    entropy_total is the sum of the three pools' scalar attention-entropy
    terms (see EntityAttentionPool.forward), used by the training loop's
    regularizer; every other caller simply ignores it."""
    pooled, weights, entropies = {}, {}, {}
    pooled["cpo"], weights["cpo"], entropies["cpo"] = model.cpo_pool(*ctx_hist["cpo"])
    pooled["emsp"], weights["emsp"], entropies["emsp"] = model.emsp_pool(*ctx_hist["emsp"])
    pooled["ev"], weights["ev"], entropies["ev"] = model.ev_pool(*ctx_hist["ev"])

    e_cpo = pooled["cpo"][scored_idx["cpo"]]
    e_emsp = pooled["emsp"][scored_idx["emsp"]]
    e_ev = pooled["ev"][scored_idx["ev"]]
    logits = model(scored_X, e_cpo, e_emsp, e_ev)
    entropy_total = entropies["cpo"] + entropies["emsp"] + entropies["ev"]
    return logits, weights, entropy_total


def get_or_train_model(df, day_cutoff):
    """CHANGED in v2: returns a 4-tuple (model, scaler, device,
    excluded_features) instead of v1's 3-tuple. On a fresh (no existing
    checkpoint) run, also computes and prints the automatic leak-feature
    detection (resolve_excluded_features) before building any entity
    history, and adds the attention-entropy regularization term to the
    training loss. See module docstring."""
    model_path = model_checkpoint_path()
    scaler_path = scaler_checkpoint_path()
    exclusions_path = exclusions_checkpoint_path()
    device = _device()
    session_dim = len(ml_features.ALL_FEATURES)
    feature_dim = len(ml_features.ALL_FEATURES)
    model = SessionTransformer(session_dim, feature_dim, MAX_NEIGHBOR_HISTORY).to(device)

    if os.path.exists(model_path) and os.path.exists(scaler_path) and os.path.exists(exclusions_path):
        print(f"[{MODEL_NAME}] checkpoint already exists -- loading, not retraining (device={device})")
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        scaler = joblib.load(scaler_path)
        excluded_features = joblib.load(exclusions_path)
        return model, scaler, device, excluded_features

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    full_train_df = df[(df["variant"] == TRAIN_VARIANT) & (df["day_idx"] < day_cutoff)].copy()
    val_cutoff = _train_val_day_cutoff(full_train_df)
    sub_train_df = full_train_df[full_train_df["day_idx"] < val_cutoff].copy()
    val_df = full_train_df[full_train_df["day_idx"] >= val_cutoff].copy()

    scaler = StandardScaler().fit(sub_train_df[ml_features.ALL_FEATURES])
    sub_train_df[ml_features.ALL_FEATURES] = scaler.transform(sub_train_df[ml_features.ALL_FEATURES])
    val_df[ml_features.ALL_FEATURES] = scaler.transform(val_df[ml_features.ALL_FEATURES])

    excluded_features, feature_fstats = resolve_excluded_features(sub_train_df, ml_features.ALL_FEATURES)
    print(f"=== Automatic leak-feature detection (ANOVA F-statistic, '{TRAIN_VARIANT}' sub_train) ===")
    for key in ENTITY_COLS:
        ref = LEAK_REFERENCE_FEATURE.get(key)
        top5 = sorted(feature_fstats[key].items(), key=lambda kv: -kv[1])[:5]
        top5_str = ", ".join(f"{f}={v:.1f}" for f, v in top5)
        print(f"  [{key}] reference leaker: {ref or '(none known -- no auto-exclusion applied for this axis)'}")
        print(f"       top-5 F-statistics: {top5_str}")
        print(f"       resolved exclusions for this axis: {excluded_features.get(key, [])}")

    os.makedirs(OUT_DIR, exist_ok=True)
    joblib.dump(excluded_features, exclusions_path)

    entity_maps = _build_entity_maps(sub_train_df)
    ctx_hist = _to_device(_entity_history_tensors(sub_train_df, entity_maps, excluded_features), device)

    sub_train_idx = {k: torch.tensor(v, dtype=torch.long, device=device)
                      for k, v in _entity_indices(sub_train_df, entity_maps).items()}
    val_idx = {k: torch.tensor(v, dtype=torch.long, device=device)
               for k, v in _entity_indices(val_df, entity_maps).items()}

    X_sub_train = torch.tensor(sub_train_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
    X_val = torch.tensor(val_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
    y_sub_train = sub_train_df[ml_features.TARGET].to_numpy().astype(np.float32)
    y_val = val_df[ml_features.TARGET].to_numpy()
    y_sub_train_t = torch.tensor(y_sub_train, device=device)

    n_pos = float(y_sub_train.sum())
    n_neg = float(len(y_sub_train) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_auroc = -1.0
    best_state = None
    epochs_without_improvement = 0
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits_train, _, entropy_total = _forward_logits(model, X_sub_train, sub_train_idx, ctx_hist, device)
        bce_loss = criterion(logits_train, y_sub_train_t)
        loss = bce_loss - ATTN_ENTROPY_WEIGHT * entropy_total
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_val, _, _ = _forward_logits(model, X_val, val_idx, ctx_hist, device)
            val_proba = torch.sigmoid(logits_val).cpu().numpy()
        val_auroc = roc_auc_score(y_val, val_proba)

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0 or epochs_without_improvement >= PATIENCE:
            print(f"  epoch {epoch:3d}  bce={bce_loss.item():.4f}  entropy={entropy_total.item():.4f}  "
                  f"loss={loss.item():.4f}  val_auroc={val_auroc:.4f}  "
                  f"best={best_val_auroc:.4f}  no_improve={epochs_without_improvement}")

        if epochs_without_improvement >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (no val AUROC improvement for {PATIENCE} epochs)")
            break

    model.load_state_dict(best_state)
    model.eval()

    torch.save(model.state_dict(), model_path)
    joblib.dump(scaler, scaler_path)
    joblib.dump(excluded_features, exclusions_path)
    print(f"[{MODEL_NAME}] trained on '{TRAIN_VARIANT}' only: n_sub_train={len(sub_train_df)}  "
          f"n_val={len(val_df)}  best_val_auroc={best_val_auroc:.4f}  "
          f"({time.time() - t0:.1f}s, device={device}) -> {model_path}")
    return model, scaler, device, excluded_features


def evaluate_on_variant(model, scaler, device, df, day_cutoff, variant_name, excluded_features,
                          zero_own_features=False):
    """Same contract as v1, with `excluded_features` now an explicit
    required argument (see module docstring) threaded into entity-history
    construction."""
    train_df = df[(df["variant"] == variant_name) & (df["day_idx"] < day_cutoff)].copy()
    test_df = df[(df["variant"] == variant_name) & (df["day_idx"] >= day_cutoff)].copy()

    train_df[ml_features.ALL_FEATURES] = scaler.transform(train_df[ml_features.ALL_FEATURES])
    test_df[ml_features.ALL_FEATURES] = scaler.transform(test_df[ml_features.ALL_FEATURES])

    entity_maps = _build_entity_maps(train_df)
    ctx_hist = _to_device(_entity_history_tensors(train_df, entity_maps, excluded_features), device)
    test_idx = {k: torch.tensor(v, dtype=torch.long, device=device)
                for k, v in _entity_indices(test_df, entity_maps).items()}

    X_test_np = test_df[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32)
    if zero_own_features:
        X_test_np = np.zeros_like(X_test_np)
    X_test = torch.tensor(X_test_np, device=device)
    y_test = test_df[ml_features.TARGET].to_numpy()

    model.eval()
    with torch.no_grad():
        logits_test, _, _ = _forward_logits(model, X_test, test_idx, ctx_hist, device)
    proba_risk = torch.sigmoid(logits_test).cpu().numpy()
    trust_score = 1.0 - proba_risk

    auroc = roc_auc_score(y_test, proba_risk)
    auprc = average_precision_score(y_test, proba_risk)

    if not zero_own_features:
        decisions = [trust_weighted_decision(1.0, float(t)).value for t in trust_score]
        preds_path = variant_preds_path(variant_name)
        out = test_df[["session_id", "variant", "world_id", "cpo_id", "emsp_id", "ev_id", "day_idx",
                        "risk_factor_count", "any_risk"]].copy()
        out["trust_score"] = trust_score
        out["decision"] = decisions
        out.to_csv(preds_path, index=False)

    return {"model": MODEL_NAME, "variant": variant_name,
            "auroc": float(auroc), "auprc": float(auprc), "n_test": int(len(test_df))}


def entity_only_ablation(model, scaler, device, df, day_cutoff, variants, excluded_features):
    """Same methodology as v1, with `excluded_features` threaded through and
    the output filename now parameterized by MODEL_NAME (v1 used the FIXED
    name 'transformer_entity_only_ablation.csv' -- that file is untouched by
    this run, so it remains available as the 'before' side of the
    before/after comparison in the notebook)."""
    rows = []
    axis_cols = {"overall": None, "cpo": "gt_cpo_malicious", "emsp": "gt_emsp_malicious", "ev": "gt_ev_sybil"}

    for variant_name in variants:
        test_df = df[(df["variant"] == variant_name) & (df["day_idx"] >= day_cutoff)].copy()
        train_df = df[(df["variant"] == variant_name) & (df["day_idx"] < day_cutoff)].copy()

        train_scaled = train_df.copy()
        train_scaled[ml_features.ALL_FEATURES] = scaler.transform(train_scaled[ml_features.ALL_FEATURES])
        test_scaled = test_df.copy()
        test_scaled[ml_features.ALL_FEATURES] = scaler.transform(test_scaled[ml_features.ALL_FEATURES])

        entity_maps = _build_entity_maps(train_scaled)
        ctx_hist = _to_device(_entity_history_tensors(train_scaled, entity_maps, excluded_features), device)
        test_idx = {k: torch.tensor(v, dtype=torch.long, device=device)
                    for k, v in _entity_indices(test_scaled, entity_maps).items()}

        X_real = torch.tensor(test_scaled[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)
        X_ablated = torch.zeros_like(X_real)

        model.eval()
        with torch.no_grad():
            proba_real = torch.sigmoid(_forward_logits(model, X_real, test_idx, ctx_hist, device)[0]).cpu().numpy()
            proba_ablated = torch.sigmoid(
                _forward_logits(model, X_ablated, test_idx, ctx_hist, device)[0]).cpu().numpy()

        row = {"variant": variant_name, "n_test": int(len(test_df))}
        for axis, col in axis_cols.items():
            y = test_df["any_risk"].to_numpy() if col is None else test_df[col].to_numpy()
            if y.sum() == 0 or y.sum() == len(y):
                row[f"real_{axis}_auroc"] = float("nan")
                row[f"ablated_{axis}_auroc"] = float("nan")
                row[f"delta_{axis}_auroc"] = float("nan")
                continue
            real_auroc = float(roc_auc_score(y, proba_real))
            ablated_auroc = float(roc_auc_score(y, proba_ablated))
            row[f"real_{axis}_auroc"] = real_auroc
            row[f"ablated_{axis}_auroc"] = ablated_auroc
            row[f"delta_{axis}_auroc"] = ablated_auroc - real_auroc
        rows.append(row)

    out = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, f"{MODEL_NAME}_entity_only_ablation.csv")
    os.makedirs(OUT_DIR, exist_ok=True)
    out.to_csv(out_path, index=False)
    print(f"Saved entity-only ablation -> {out_path}")
    return out


def score_with_attention(model, scaler, device, df, day_cutoff, variant_name, excluded_features):
    """Same purpose as v1, with `excluded_features` threaded through and the
    v1 `.cpu()`-on-a-numpy-array bug (test_idx_np is already numpy; no
    .cpu() call needed or valid on it) fixed by construction -- test_idx_np
    is now returned directly."""
    train_df = df[(df["variant"] == variant_name) & (df["day_idx"] < day_cutoff)].copy()
    test_df = df[(df["variant"] == variant_name) & (df["day_idx"] >= day_cutoff)].copy()

    train_scaled = train_df.copy()
    train_scaled[ml_features.ALL_FEATURES] = scaler.transform(train_scaled[ml_features.ALL_FEATURES])
    test_scaled = test_df.copy()
    test_scaled[ml_features.ALL_FEATURES] = scaler.transform(test_scaled[ml_features.ALL_FEATURES])

    entity_maps = _build_entity_maps(train_scaled)
    ctx_hist = _to_device(_entity_history_tensors(train_scaled, entity_maps, excluded_features), device)
    test_idx_np = _entity_indices(test_scaled, entity_maps)
    test_idx = {k: torch.tensor(v, dtype=torch.long, device=device) for k, v in test_idx_np.items()}
    X_test = torch.tensor(test_scaled[ml_features.ALL_FEATURES].to_numpy(dtype=np.float32), device=device)

    model.eval()
    with torch.no_grad():
        logits_test, attn_weights, _ = _forward_logits(model, X_test, test_idx, ctx_hist, device)
    proba_risk = torch.sigmoid(logits_test).cpu().numpy()

    return {
        "test_df": test_df.reset_index(drop=True),
        "test_idx": test_idx_np,
        "attn_weights": {k: v.cpu().numpy() for k, v in attn_weights.items()},
        "entity_maps": entity_maps,
        "train_df": train_df,
        "proba_risk": proba_risk,
    }


def entity_recent_window(context_df, entity_type, entity_value, max_history=MAX_NEIGHBOR_HISTORY):
    col = ENTITY_COLS[entity_type]
    rows = context_df[context_df[col] == entity_value].sort_values("day_idx", ascending=False).head(max_history)
    return rows.sort_values("day_idx", ascending=True).reset_index(drop=True)


def main(combined_csv_path):
    """Return contract UNCHANGED from v1 (returns just `summary`, matching
    every other baseline) -- the notebook calls get_or_train_model()
    directly, separately, wherever it needs excluded_features (same pattern
    v1's notebook already used to get model/scaler/device for the ablation
    and visualization steps)."""
    df, day_cutoff = _prepare_full_frame(combined_csv_path)
    variants = sorted(df["variant"].unique()) if "variant" in df.columns else [None]
    device = _device()
    print(f"day_cutoff={day_cutoff}  variants found: {variants}")
    print(f"Training 1 model ({MODEL_NAME}) on '{TRAIN_VARIANT}' only, then evaluating "
          f"zero-shot on every variant's own test split (device={device})")

    model, scaler, device, excluded_features = get_or_train_model(df, day_cutoff)

    all_rows = []
    for variant_name in variants:
        row = evaluate_on_variant(model, scaler, device, df, day_cutoff, variant_name, excluded_features)
        all_rows.append(row)
        print(f"  [{MODEL_NAME:20s} -> {variant_name:16s}] "
              f"AUROC={row['auroc']:.4f}  AUPRC={row['auprc']:.4f}  n_test={row['n_test']}")

    summary = pd.DataFrame(all_rows)
    summary_path = os.path.join(OUT_DIR, f"{MODEL_NAME}_variant_summary.csv")
    os.makedirs(OUT_DIR, exist_ok=True)
    summary.to_csv(summary_path, index=False)

    print(f"\n=== Summary table (trained once on '{TRAIN_VARIANT}', zero-shot evaluated "
          f"per variant) -- saved to {summary_path} ===")
    print(summary.set_index("variant")[["auroc", "auprc"]].round(4).to_string())

    seed_variants = [v for v in variants if v.startswith("seed_")]
    if seed_variants:
        print(f"\n=== Stability check across {len(seed_variants)} independent reseeds "
              f"({', '.join(seed_variants)}) -- mean +/- std AUROC ===")
        stab = summary[summary["variant"].isin(seed_variants)]["auroc"]
        print(f"  mean={stab.mean():.4f}  std={stab.std():.4f}")

    return summary


if __name__ == "__main__":
    combined_csv_path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    main(combined_csv_path)


## Step 2 -- train once on `main`, evaluate zero-shot on every variant

Same protocol as every other baseline. Because `MODEL_NAME` changed (`transformer_lite_v2`), this trains a genuinely fresh model -- it will NOT silently reload v1's checkpoint. Watch the printed automatic-leak-detection block at the start of training: it shows every feature's F-statistic per entity type and which ones got excluded as a result.

In [ ]:
import os
import glob

# Force retraining by removing existing checkpoint files for transformer_lite_v2
out_dir = os.environ.get('V2G_TRANSFORMER_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/transformer_baseline')
checkpoint_files = glob.glob(os.path.join(out_dir, '*transformer_lite_v2*'))

if checkpoint_files:
    print(f"Removing existing checkpoints to force retraining:")
    for f in checkpoint_files:
        os.remove(f)
        print(f"  Deleted: {f}")
else:
    print("No existing transformer_lite_v2 checkpoints found. Training will proceed fresh.")


In [ ]:
import importlib
import transformer_baseline as tb
importlib.reload(tb)

summary = tb.main(combined_csv_path)
summary

## Step 3 -- entity-only ablation: before (v1) vs. after (v2) vs. graph_baseline

Re-runs the same entity-only ablation methodology as v1 (and as `graph_baseline.py`'s own ablation), then prints THREE results side by side for the same variants:
- **v1 (before the fix)** -- loaded from v1's untouched `transformer_entity_only_ablation.csv` (fixed filename, not overwritten by this run).
- **v2 (after the fix)** -- this run's `transformer_lite_v2_entity_only_ablation.csv`.
- **graph_baseline (mean-pooling, already fixed)** -- `graph_baseline_entity_only_ablation.csv`, if present on Drive.

**How to read the comparison:** look specifically at `delta_cpo_auroc` and `delta_emsp_auroc`. If v2's CPO delta moved from near-zero toward something resembling graph_baseline's (large, negative), the fix did something real on the one axis where a real behavioral signal is possible (CPO has onset_day time-gating). The eMSP delta may legitimately stay near-zero regardless -- see Step 4, which shows directly why no model can do otherwise on that specific axis as currently labeled. Also check `delta_ev_auroc` didn't shrink much from v1 to v2 -- if the entropy regularizer destroyed EV's genuine signal along with the shortcut, that's a real cost worth knowing about, not something to gloss over.

In [ ]:
import pandas as pd

df, day_cutoff = tb._prepare_full_frame(combined_csv_path)
model, scaler, device, excluded_features = tb.get_or_train_model(df, day_cutoff)  # loads Step 2's checkpoint

variants = sorted(df['variant'].unique())
ablation_v2 = tb.entity_only_ablation(model, scaler, device, df, day_cutoff, variants, excluded_features)

display_cols = ['variant', 'n_test'] + [c for c in ablation_v2.columns if c not in ('variant', 'n_test')]

print('=== v2 (after the fix): transformer entity-only ablation ===')
print(ablation_v2[display_cols].round(4).to_string(index=False))

out_base = os.environ['V2G_TRANSFORMER_OUT_DIR']
v1_path = f'{out_base}/transformer_entity_only_ablation.csv'
graph_path = f"{out_base.rsplit('/', 1)[0]}/graph_baseline/graph_baseline_entity_only_ablation.csv"

try:
    ablation_v1 = pd.read_csv(v1_path)
    print(f"\n=== v1 (before the fix) -- loaded from {v1_path} ===")
    print(ablation_v1[[c for c in display_cols if c in ablation_v1.columns]].round(4).to_string(index=False))

    print('\n=== v1 -> v2 change in the axis deltas that matter (delta_cpo_auroc, delta_emsp_auroc, delta_ev_auroc) ===')
    merged = ablation_v1.merge(ablation_v2, on='variant', suffixes=('_v1', '_v2'))
    for axis in ['cpo', 'emsp', 'ev']:
        col = f'delta_{axis}_auroc'
        if f'{col}_v1' in merged.columns and f'{col}_v2' in merged.columns:
            print(f"\n  {axis.upper()} axis:")
            print(merged[['variant', f'{col}_v1', f'{col}_v2']].round(4).to_string(index=False))
except FileNotFoundError:
    print(f'\n(No v1-vs-v2 comparison: {v1_path} not found -- that file should exist from v1\'s own run.)')

try:
    ablation_graph = pd.read_csv(graph_path)
    print(f'\n=== graph_baseline (mean-pooling, already fixed) -- loaded from {graph_path} ===')
    print(ablation_graph.round(4).to_string(index=False))
except FileNotFoundError:
    print(f'\n(No graph_baseline comparison: {graph_path} not found.)')

print('\nReading: a SMALL |delta| on an axis means the entity-attention pathway alone nearly')
print('reproduces that axis\'s AUROC (identity-shortcut signature). A LARGE negative delta means')
print('the session\'s own features are doing real work -- the behavioral, non-shortcut pattern.')

## Step 4 -- a genuinely informative attention check

v1's qualitative plot picked the *first* malicious test example for each axis and hoped its history window showed contrast between honest and malicious rows -- for both the CPO and eMSP panels it actually picked up, it happened to land on a window that was 100% malicious, which isn't a plot that can show anything (there's nothing to contrast). At the time, that was structurally guaranteed for eMSP (no onset-gating existed) and merely unlucky for CPO. Since then, the dataset itself was fixed (see this notebook's intro): both `gt_cpo_malicious` and `gt_emsp_malicious` now have a real, independent onset day, so this cell searches BOTH axes symmetrically for a test example whose entity's context-split window actually contains both honest and malicious rows, instead of taking the first candidate blindly or assuming one axis can never show contrast. If a search still comes up empty for one axis on a given variant, that's now a real, reportable finding (e.g. the onset day landed too close to this variant's train/test boundary for that particular entity), not an expected structural dead end -- the printout below says so plainly rather than papering over it.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

scored = tb.score_with_attention(model, scaler, device, df, day_cutoff, 'main', excluded_features)
train_df = scored['train_df']  # raw (unscaled) train rows

POOLS = {'cpo': model.cpo_pool, 'emsp': model.emsp_pool, 'ev': model.ev_pool}
feature_cols = tb.ml_features.ALL_FEATURES

def plot_transition_attention(entity_type, entity_val, gt_col, ax):
    entity_col = tb.ENTITY_COLS[entity_type]
    sub = train_df[train_df[entity_col] == entity_val].sort_values('day_idx').reset_index(drop=True)

    labels = sub[gt_col].to_numpy()
    transitions = np.where(labels[:-1] != labels[1:])[0]
    if len(transitions) == 0:
        ax.set_title(f"{entity_type.upper()} {entity_val}: No transition found")
        return

    trans_idx = transitions[0]
    start = max(0, trans_idx - 12)
    end = min(len(sub), start + tb.MAX_NEIGHBOR_HISTORY)
    window = sub.iloc[start:end].reset_index(drop=True)
    k = len(window)

    # Build THIS window's own feature tensor (same scaler the model trained with) and
    # run it through the entity's real, frozen attention pool directly -- so the weights
    # plotted are actually computed over these rows, not reused from the cached
    # end-of-train snapshot (a different, much-later slice of this entity's history).
    vals = scaler.transform(window[feature_cols]).astype(np.float32)
    excluded_idx = [feature_cols.index(f) for f in excluded_features.get(entity_type, [])]
    if excluded_idx:
        vals[:, excluded_idx] = 0.0

    hist = np.zeros((1, tb.MAX_NEIGHBOR_HISTORY, len(feature_cols)), dtype=np.float32)
    mask = np.zeros((1, tb.MAX_NEIGHBOR_HISTORY), dtype=bool)
    hist[0, tb.MAX_NEIGHBOR_HISTORY - k:, :] = vals
    mask[0, tb.MAX_NEIGHBOR_HISTORY - k:] = True

    pool = POOLS[entity_type]
    pool.eval()
    with torch.no_grad():
        _, weights_summary, _ = pool(torch.tensor(hist, device=device), torch.tensor(mask, device=device))
    weights_slice = weights_summary[0].cpu().numpy()[tb.MAX_NEIGHBOR_HISTORY - k:]

    colors = ['#C44E52' if v else '#4C72B0' for v in window[gt_col].to_numpy().astype(bool)]
    x = np.arange(k)
    ax.bar(x, weights_slice, color=colors)
    ax.set_title(f'{entity_type.upper()} ({entity_val}) Transition Window\n(Blue = Honest, Red = Malicious)')
    ax.set_xlabel('Sequence position in history window')
    ax.set_ylabel('Attention Weight')
    ax.set_xticks(x)
    ax.set_xticklabels(window['day_idx'].to_numpy(), rotation=45)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_transition_attention('cpo', 'CPO-B', 'gt_cpo_malicious', axes[0])
plot_transition_attention('emsp', 'eMSP-C', 'gt_emsp_malicious', axes[1])
red_patch = plt.matplotlib.patches.Patch(color='#C44E52', label='Malicious Session')
blue_patch = plt.matplotlib.patches.Patch(color='#4C72B0', label='Honest Session')
fig.legend(handles=[red_patch, blue_patch], loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.05))
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let's inspect the actual transitions in train_df to find why the strict window check failed
# and plot a historical window that highlights the transition for both CPO and eMSP.
for entity, gt_col in [('cpo', 'gt_cpo_malicious'), ('emsp', 'gt_emsp_malicious')]:
    col = tb.ENTITY_COLS[entity]
    print(f"\n--- {entity.upper()} transition search ---")
    for val in train_df[col].unique():
        sub = train_df[train_df[col] == val].sort_values('day_idx')
        uniques = sub[gt_col].unique()
        print(f"  Entity {val}: unique labels {uniques}, total rows {len(sub)}")

## Step 5 -- EV-axis isolated comparison

The EV axis has no N=3-style low-cardinality identity risk (~1000 distinct EVs) and v1's own ablation already showed a real, negative delta there (own features matter). This cell pulls just the EV-axis real/ablated/delta numbers across variants from v2's ablation and prints them against graph_baseline's own EV numbers -- the one comparison between the two pooling mechanisms that isn't contaminated by the CPO/eMSP shortcut risk either way.

In [ ]:
ev_cols = ['variant', 'n_test', 'real_ev_auroc', 'ablated_ev_auroc', 'delta_ev_auroc']

print('=== v2 transformer, EV axis only ===')
print(ablation_v2[ev_cols].round(4).to_string(index=False))

try:
    ablation_graph = pd.read_csv(graph_path)
    # graph_baseline.py's ablation CSV uses a DIFFERENT column-naming convention
    # ('real_ev_axis_auroc'/'ablated_ev_axis_auroc', no 'n_test', no precomputed
    # delta) than transformer_baseline.py's ('real_ev_auroc'/'ablated_ev_auroc'/
    # 'delta_ev_auroc') -- the naive column filter here used to silently match only
    # 'variant' against the wrong names instead of erroring, which is why this
    # comparison printed an empty table. Renamed explicitly instead of assuming
    # the two modules share a schema.
    graph_ev = ablation_graph[['variant', 'real_ev_axis_auroc', 'ablated_ev_axis_auroc']].copy()
    graph_ev['delta_ev_auroc'] = graph_ev['ablated_ev_axis_auroc'] - graph_ev['real_ev_axis_auroc']
    graph_ev = graph_ev.rename(columns={'real_ev_axis_auroc': 'real_ev_auroc',
                                         'ablated_ev_axis_auroc': 'ablated_ev_auroc'})
    print(f'\n=== graph_baseline (mean-pooling), EV axis only -- for comparison ===')
    print(graph_ev[['variant', 'real_ev_auroc', 'ablated_ev_auroc', 'delta_ev_auroc']].round(4).to_string(index=False))
except FileNotFoundError:
    print(f'\n(No graph_baseline EV comparison: {graph_path} not found.)')

print('\nBoth pooling mechanisms should show a real, negative delta_ev_auroc here (own features')
print('matter) -- this axis never showed the identity-shortcut signature under either mechanism,')
print('so the main thing worth checking is whether v2\'s entropy regularizer eroded this delta')
print('noticeably relative to v1\'s (see Step 3\'s before/after table) -- if it did, the')
print('regularization strength (ATTN_ENTROPY_WEIGHT) is costing genuine signal, not just leaks.')

## Where this leaves things

Not a verdict written in advance -- read Step 3's before/after table and Step 4's printout to see what actually happened on this run. CPO and eMSP are now judged by the SAME standard, since both have real onset-gated ground truth:

1. **CPO and eMSP axes** -- did `delta_cpo_auroc` and `delta_emsp_auroc` move toward graph_baseline's pattern (real, negative -- own features matter) from v1's near-zero-on-both? If yes for both, the dataset's onset-gating fix and the model-side auto-exclusion/entropy regularization did something real together. If one axis moved and the other didn't, that's informative on its own: entity-aggregation still finds an identity-adjacent shortcut on whichever axis didn't move, even with genuine behavioral signal now available to learn instead -- worth digging into directly (the same way `previous_flag_rate` was originally found for eMSP) rather than assumed away.
2. **EV axis** -- did `delta_ev_auroc` survive roughly intact from v1 to v2 (Step 3's comparison, Step 5's isolated view)? If it shrank a lot, the entropy regularizer is too strong and is taking out genuine signal along with the shortcut.
3. **Step 4's search** -- did it find a real contrast window for both CPO and eMSP this time? If one came up empty, check the printed fraction-malicious breakdown: it's expected only if that axis's onset day happens to fall outside this variant's context-split day range, not otherwise.
4. Only once both CPO and eMSP look behavioral and EV's genuine delta survives intact does a seed-stability sweep (`seed_101/103/104`) become the next real gate before considering this for the baseline ladder -- not done here.